In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

In [2]:
risk_df = pd.read_csv("../outputs/risk_metrics.csv")
signals_df = pd.read_csv("../data/predicted_portfolio_signals.csv")
risk_df = risk_df[risk_df["Stock"] != "NIFTY50_all"].reset_index(drop=True)
ai_pred_returns = signals_df.groupby("Name")["Predicted_Return"].mean().reset_index()
master_pool = risk_df.merge(
    ai_pred_returns, left_on="Stock", right_on="Name", how="inner"
)

In [3]:
def filter_by_profile(df, profile):
    if profile == "Conservative":
        # low risk, positive sharpe ratio
        return df[
            (df["Volatility"] < df["Volatility"].quantile(0.40)) & (df["Sharpe"] > 0)
        ]

    elif profile == "Balanced":
        # moderate risk, positive sortino ratio
        return df[
            (df["Volatility"] < df["Volatility"].quantile(0.70)) & (df["Sortino"] > 0)
        ]

    elif profile == "Aggressive":
        # high risk, exclude stocks with extreme historic losses
        return df[
            (df["Predicted_Return"] > df["Predicted_Return"].quantile(0.7))
        ]

    return df

In [4]:
def get_optimization_inputs(filtered_df):
    universal_df = pd.read_csv("../data/universal_training_data.csv")
    valid_tickers = filtered_df["Stock"].tolist()
    price_history = universal_df[universal_df["Name"].isin(valid_tickers)].copy()
    price_history["Timeline_Index"] = price_history.groupby("Name").cumcount()

    matrix_pivot = price_history.pivot(
        index="Timeline_Index", columns="Name", values="Daily_Return"
    ).dropna()
    cov_matrix = matrix_pivot.cov() * 252

    aligned_stocks = [s for s in valid_tickers if s in cov_matrix.index]
    cov_matrix = cov_matrix.loc[aligned_stocks, aligned_stocks]
    expected_returns = (
        ai_pred_returns.set_index("Name").loc[aligned_stocks, "Predicted_Return"].values
    )

    return aligned_stocks, cov_matrix, expected_returns, len(aligned_stocks)

In [5]:
# Portfolio Variance Formula: W^T * Covariance_Matrix * W
def get_portfolio_variance(weights, cov_mat):
    return np.dot(weights.T, np.dot(cov_mat, weights))


# Sharpe Ratio Maximization Function
def get_neg_sharpe(weights, exp_returns, cov_mat, risk_free_rate=0.06):
    p_return = np.sum(exp_returns * weights) * 252  # Expected annualized return
    p_vol = np.sqrt(get_portfolio_variance(weights, cov_mat))
    return -(p_return - risk_free_rate) / p_vol


# Direct Return Maximization Function
def get_neg_return(weights, exp_returns):
    return -np.sum(exp_returns * weights)


profile_outputs = []

for profile_name in ["Conservative", "Balanced", "Aggressive"]:
    screened_pool = filter_by_profile(master_pool, profile_name)
    stocks, cov_mat, exp_ret, num_assets = get_optimization_inputs(screened_pool)

    #  Total allocated weights must sum upto 100%
    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}

    #  No short-selling (w >= 0), single stock should not exceed 20% of the capital
    bounds = tuple((0.0, 0.20) for _ in range(num_assets))
    initial_guess = num_assets * [1.0 / num_assets]

    if profile_name == "Conservative":
        res = minimize(
            get_portfolio_variance,
            initial_guess,
            args=(cov_mat,),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
        )
    elif profile_name == "Balanced":
        res = minimize(
            get_neg_sharpe,
            initial_guess,
            args=(exp_ret, cov_mat),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
        )
    elif profile_name == "Aggressive":
        res = minimize(
            get_neg_return,
            initial_guess,
            args=(exp_ret,),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
        )

    df_res = pd.DataFrame(
        {
            "Stock": stocks,
            "Allocation_%": np.round(res.x * 100, 2),
            "Profile": profile_name,
        }
    )
    df_res = df_res.merge(
        master_pool[
            [
                "Stock",
                "Predicted_Return",
                "Sharpe",
                "Sortino",
                "Volatility",
                "Max_Drawdown",
            ]
        ],
        on="Stock",
        how="left",
    )
    profile_outputs.append(df_res)
    final_allocations = pd.concat(profile_outputs, ignore_index=True)

    # Only keeping stocks that received an allocation higher than 0.5%
    final_allocations = final_allocations[
        final_allocations["Allocation_%"] > 0.5
    ].reset_index(drop=True)

    final_allocations.to_csv("../outputs/portfolio_allocations.csv", index=False)